## Pratiksha Dhembare 
## 260240128031

## Vijay Rajage
## 260240128053


# ASSIGNMENT NUMBER : 04

In [1]:
import requests
from bs4 import BeautifulSoup
from collections import Counter
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer

In [2]:
def isReal(num):
    try:
        float(num)
        return True
    except ValueError:
        return False

# Open the wikipedia page https://en.wikipedia.org/wiki/Saturn
url = "https://en.wikipedia.org/wiki/Saturn"
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

paragraphs = soup.find_all('p')
saturn_text = " ".join([p.text for p in paragraphs])

clean_pune = [word for word in word_tokenize(saturn_text) if word.isalpha() or isReal(word)]
pos_pune = pos_tag(clean_pune)

### 1. Scrap the web and find top 10 nouns from the page

In [3]:
noun_list = []
for word, tag in pos_pune:
    if tag.startswith("NN"):
        noun_list.append(word)
top_10_nouns = Counter(noun_list).most_common(10)

In [ ]:

print("Top 10 Nouns")
for noun, count in top_10_nouns:
    print(f"{noun}: {count}")

--- Top 10 Nouns ---
Saturn: 134
planet: 31
rings: 29
Earth: 27
System: 21
Jupiter: 20
Titan: 19
Cassini: 19
rotation: 18
Sun: 17


### 2. Print all the unique verbs in their root form.

In [5]:
 
lemmatizer = WordNetLemmatizer()
verb_list = []

In [6]:
for word, tag in pos_pune:
    if tag.startswith("VB"):
        root_form = lemmatizer.lemmatize(word.lower(), pos='v')
        verb_list.append(root_form)

unique_verbs = sorted(list(set(verb_list)))

In [23]:
print("Unique Verbs (Root Form)")
print(unique_verbs)


Unique Verbs (Root Form)
['accumulate', 'achieve', 'acquire', 'add', 'allow', 'ammonia', 'angle', 'announce', 'appear', 'approach', 'arrive', 'assign', 'associate', 'assume', 'base', 'be', 'become', 'begin', 'believe', 'bland', 'briefly', 'bulge', 'call', 'capture', 'carry', 'cause', 'change', 'charge', 'christianize', 'claim', 'classify', 'come', 'compare', 'complete', 'compose', 'comprise', 'condense', 'conduct', 'confine', 'confirm', 'consider', 'consist', 'contain', 'continue', 'correspond', 'cover', 'create', 'define', 'deflect', 'depend', 'depict', 'deplete', 'derive', 'descend', 'describe', 'design', 'designate', 'detect', 'direct', 'disappear', 'discern', 'discover', 'display', 'divide', 'do', 'emerge', 'emit', 'encompass', 'end', 'enlarge', 'enter', 'erupt', 'estimate', 'exclude', 'exhibit', 'exist', 'expel', 'explain', 'explore', 'extend', 'feature', 'find', 'finish', 'flatten', 'follow', 'form', 'gap', 'generate', 'give', 'have', 'hold', 'hypothesize', 'identify', 'image', '

### 3. Extract all the DT JJ NN phrases from the article.

In [8]:
phrase_list = []

for idx in range(len(pos_pune) - 2):
    if (pos_pune[idx][1] == "DT" and 
        pos_pune[idx+1][1].startswith("JJ") and 
        pos_pune[idx+2][1].startswith("NN")):
        
        phrase = f"{pos_pune[idx][0]} {pos_pune[idx+1][0]} {pos_pune[idx+2][0]}"
        phrase_list.append(phrase)

unique_phrases = sorted(list(set(phrase_list)))

In [9]:
print("Extracted <DT><JJ><NN> Phrases")
for phrase in unique_phrases:
    print(phrase)

Extracted <DT><JJ><NN> Phrases
The Greek scientist
The atmospheric entry
The average distance
The brightest magnitude
The elliptical orbit
The entire structure
The hexagonal feature
The lower layers
The other side
The polar regions
The total mass
The upper clouds
The visible features
This photochemical cycle
a British team
a Greek eta
a Greek ligature
a banded pattern
a bright blue
a clear resolution
a close flyby
a deep layer
a definite surface
a destroyed moon
a gaseous planet
a great angel
a high phase
a horizontal stroke
a liquid layer
a magnetic dipole
a magnetic moment
a major atmosphere
a major character
a near resonance
a new moon
a peppered coating
a possible precursor
a potential habitat
a retrograde orbit
a rocky core
a significant fraction
a small rocky
a small telescope
a smaller amount
a standard deviation
a substantial atmosphere
a tenuous ring
a total lifecycle
a warm polar
all giant storms
an average radius
an entire circuit
an indeterminate gradient
an indicated rotat

### 4. Summarize the text using TextRank Algorithm

In [10]:
import nltk
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [11]:
sentences = nltk.sent_tokenize(saturn_text)

In [12]:
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(sentences)

In [13]:
similarity_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [14]:
graph = nx.from_numpy_array(similarity_matrix)

In [15]:
scores = nx.pagerank(graph)

In [16]:
ranked_sentences = sorted(((scores[i], s) for i, s in enumerate(sentences)), reverse=True)

In [17]:
textrank_summary = [ranked_sentences[i][1] for i in range(min(3, len(sentences)))]
print(" TextRank Summary: ")
print(" ".join(textrank_summary))

 TextRank Summary: 

 Saturn is the sixth planet from the Sun and the second largest in the Solar System, after Jupiter. Huygens discovered Saturn's moon Titan. Even though Saturn is almost as big as Jupiter, Saturn has less than a third of its mass.


### 5. Summarize the text using extractive summarization (LexRank via Sumy)

In [18]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer

In [19]:
parser = PlaintextParser.from_string(saturn_text, Tokenizer("english"))

In [20]:
summarizer = LexRankSummarizer()

In [21]:
lexrank_summary_sentences = summarizer(parser.document, 3)

In [ ]:
print("LexRank Extractive Summary")
for sentence in lexrank_summary_sentences:
    print(sentence)

--- LexRank Extractive Summary ---
It is a gas giant, with an average radius of about 9 times that of Earth.
Titan, Saturn's largest moon and the second largest in the Solar System, is larger (but less massive) than the planet Mercury and is the only moon in the Solar System that has a substantial atmosphere.
[100] Saturn and its rings are best seen when the planet is at, or near, opposition, the configuration of a planet when it is at an elongation of 180°, and thus appears opposite the Sun in the sky.
